# 4. Feature engineering (M2)

Este notebook cria novas variáveis a partir das colunas fornecidas e mede o ganho de cada uma sobre o M1, do notebook anterior (`03_modelo_base`)

- Cada nova variável é somada à base, com 3 sementes, e avaliada no teste. Ficam as variáveis cujo ganho de AUC supera o desvio entre as sementes, ou que trazem ganho claro de estabilidade (pior KS entre as safras)
- Disclaimer: medianas de renda e taxa de inadimplência por prefixo de CEP são ajustadas só no treino

Novas features:
1. contagens de códigos especiais por linha (`9999`, `-1`,`-2`)

   Para cada cliente, conta quantas VPs vêm com `-1`, `-2` e `9999` (e também com zero e nulo)

2. renda relativa ao estado e à região

   Compara a faixa de renda do cliente com a mediana do estado e da região dele no treino, e calcula a % de clientes do estado com renda menor ou igual

3. interações renda × idade e idade × região

   Cada combinação de faixa de renda com faixa de idade vira uma categoria, e o mesmo vale para idade com região. Combinações com menos de 200 clientes no treino vão para "outros". Assim o efeito da idade pode mudar conforme a renda: ganhar até 1.000 aos 20 anos não necessariamente tem o mesmo risco que ganhar até 1.000 aos 60

4. CEP (prefixo, CEP genérico, ausente, coerência com o estado)

   CEP localiza o cliente com mais precisão que o estado. Usa a taxa de inadimplência dos 3 primeiros dígitos do CEP no treino, suavizada para a média geral quando o prefixo tem poucos clientes, e marca se o CEP termina em `000` (CEP geral da cidade), se está ausente e se é do mesmo estado informado. Criamos essas variáveis porque CEP genérico, ausente ou de outro estado pode indicar cadastro fraco

As famílias 5 a 7 ficam para depois.

## 4.1 Imports e leitura

In [ ]:
import json

import pandas as pd

from equifax.config import RANDOM_SEED, REPORTS, TABLES
from equifax.conjuntos import carregar_conjunto
from equifax.dados import carregar_desenvolvimento, janelas
from equifax.experimentos import comparar_com_referencia, ler_registro, rodar_experimento
from equifax.features import (Cep, ContagensPorLinha, Interacoes, RendaRelativa, gerar_features,
                              regiao_do_estado)
from equifax.selecao import correlacao_spearman, ficha_variaveis

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [ ]:
PARAMS_M1 = json.load(open(REPORTS / "params_m1.json", encoding="utf-8"))
CONJUNTO = PARAMS_M1["conjunto"]          # proposta_242
PARAMS_LGBM = PARAMS_M1["lgbm"]           # LightGBM ajustado do M1, fixo
VARIAVEIS_M1 = carregar_conjunto(CONJUNTO)
VPS = [v for v in VARIAVEIS_M1 if v.startswith("VP")]
print(CONJUNTO, len(VARIAVEIS_M1), "variáveis,", len(VPS), "VPs")

base = carregar_desenvolvimento()
partes = janelas(base)  # treino e teste; a OOT interna não entra

## 4.2 Gerar as famílias

Cada família aprende no treino e é aplicada ao teste. As contagens usam só as VPs do conjunto de trabalho.

In [ ]:
familias = [ContagensPorLinha(VPS), RendaRelativa(), Interacoes(minimo=200), Cep(suavizacao=200)]
partes_fe, COLUNAS = gerar_features(partes, familias)
treino, teste = partes_fe["treino"], partes_fe["teste"]
NOVAS = sum(COLUNAS.values(), [])
for familia in familias:
    print(f"{familia.nome}: {COLUNAS[familia.nome]}")

## 4.3 Ficha das variáveis novas

O mesmo critério da seleção de variáveis: IV, PSI por safra e a maior correlação de Spearman com as variáveis que já estão no modelo

In [ ]:
CATEGORICAS_NOVAS = [c for c in NOVAS if not pd.api.types.is_numeric_dtype(treino[c])]
ficha_fe, binnings_fe = ficha_variaveis(treino, NOVAS, categoricas=CATEGORICAS_NOVAS, verbose=False)

numericas = [c for c in NOVAS if c not in CATEGORICAS_NOVAS]
corr = correlacao_spearman(treino, numericas + VPS)  # pode avisar de divisão por zero (constantes)
for c in numericas:
    outras = corr.loc[c].drop(c).abs().dropna()  # variável constante não tem correlação
    ficha_fe.loc[c, "maior_correlacao"] = outras.max() if len(outras) else float("nan")
    ficha_fe.loc[c, "com_quem"] = outras.idxmax() if len(outras) else "(constante)"
ficha_fe["familia"] = [next(f for f, cols in COLUNAS.items() if v in cols) for v in ficha_fe.index]
ficha_fe.to_csv(TABLES / "ficha_features_m2.csv")
ficha_fe[["familia", "iv", "classe_iv", "psi_max", "classe_psi", "maior_correlacao", "com_quem"]].round(4)

## 4.4 Novas variáveis no M1

A coluna `supera_ruido` diz se o ganho de AUC é maior que o desvio entre sementes.

In [ ]:
EXPERIMENTOS = {"c_base_m1": VARIAVEIS_M1}
for nome, cols in COLUNAS.items():
    EXPERIMENTOS[f"c_mais_{nome}"] = VARIAVEIS_M1 + cols
# A regiao não está na base tratada: é criada aqui, só para este teste
for parte in partes_fe.values():
    parte["regiao"] = regiao_do_estado(parte["estado"])
EXPERIMENTOS["c_regiao_no_lugar_de_estado"] = ["regiao" if v == "estado" else v for v in VARIAVEIS_M1]

for nome, variaveis in EXPERIMENTOS.items():
    rodar_experimento(nome, variaveis, partes_fe, modelo="lgbm", params=PARAMS_LGBM,
                      nome_conjunto=f"{CONJUNTO}+{nome.removeprefix('c_')}")
    print(nome, "pronto")

ablacao = comparar_com_referencia(ler_registro(), "c_base_m1", [n for n in EXPERIMENTOS if n != "c_base_m1"])
ablacao.to_csv(TABLES / "ablacao_m2.csv")
ablacao

## 4.5 M2: base mais as famílias que ficaram

Entram as famílias que superaram o ruído na ablação. A troca por `regiao` fica de fora: ela é só informativa. O conjunto combinado também é medido, porque famílias podem se sobrepor.

**Regra quando nenhuma família supera o ruído (decisão da equipe, 18/10):** o regulamento exige que cada entrega gere pelo menos um modelo diferente, então o M2 não pode repetir o M1. Nesse caso entra a família de CEP: a única com hipótese de localização mais fina que o `estado`. O ganho é reportado como está.


In [ ]:
FAMILIA_SE_NENHUMA_FICAR = "f4_cep"  # regulamento: um modelo diferente em cada entrega (decisão de 18/10)

ficam = [n.removeprefix("c_mais_") for n in ablacao.index
         if n.startswith("c_mais_") and ablacao.loc[n, "supera_ruido"]]
regra = "famílias que superaram o desvio entre sementes"
if not ficam:
    ficam = [FAMILIA_SE_NENHUMA_FICAR]
    regra = "nenhuma família superou o ruído; entra a de CEP para o M2 ser um modelo diferente do M1"
print("Famílias no M2:", ficam, "|", regra)

variaveis_m2 = VARIAVEIS_M1 + sum((COLUNAS[f] for f in ficam), [])
res_m2 = rodar_experimento("c_m2_combinado", variaveis_m2, partes_fe, modelo="lgbm", params=PARAMS_LGBM,
                           nome_conjunto=f"{CONJUNTO}+m2")
comparar_com_referencia(ler_registro(), "c_base_m1", ["c_m2_combinado"])


> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

In [ ]:
# Posição das variáveis novas na importância (ganho) do modelo combinado
importancia = res_m2.importancia.sort_values(ascending=False)
posicao = importancia.rank(ascending=False).astype(int)
pd.DataFrame({"ganho": importancia, "posicao": posicao}).loc[[c for c in importancia.index if c.startswith("fe_")]].round(4)

In [ ]:
# Lista do M2, versionada (só nomes de colunas)
with open(REPORTS / "variaveis_m2.json", "w", encoding="utf-8") as f:
    json.dump({"base": CONJUNTO, "familias": ficam, "regra": regra, "variaveis": variaveis_m2}, f,
              ensure_ascii=False, indent=2)
print(len(variaveis_m2), "variáveis no M2")


## 4.6 Testes complementares: todas as famílias juntas e regressão logística

Analisando se as famílias se somam quando entram juntas, e se a regressão logística com WoE, que não cruza variáveis sozinha, ganha com as features

In [ ]:
TODAS = VARIAVEIS_M1 + NOVAS
rodar_experimento("c_mais_todas_familias", TODAS, partes_fe, modelo="lgbm", params=PARAMS_LGBM,
                  nome_conjunto=f"{CONJUNTO}+todas")
for nome, variaveis in [("c_logit_base_m1", VARIAVEIS_M1), ("c_logit_mais_todas_familias", TODAS)]:
    # a logística é determinística: uma semente; sem desvio entre sementes, supera_ruido fica vazio
    rodar_experimento(nome, variaveis, partes_fe, modelo="logit_woe", params=PARAMS_M1["logit_woe"],
                      sementes=(RANDOM_SEED,), nome_conjunto=nome)
registro = ler_registro()
display(comparar_com_referencia(registro, "c_base_m1", ["c_mais_todas_familias"]))
comparar_com_referencia(registro, "c_logit_base_m1", ["c_logit_mais_todas_familias"])

Parâmetros do M1 fixos (LightGBM ajustado do notebook `03_modelo_base`).

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.